# Exhaustive Model Training & Ensembling Pipeline

This notebook implements the exhaustive 12-model pipeline with centralized logging, Optuna hyperparameter optimization, and a custom Optuna-weighted Soft Voting Ensemble.

**Objectives:**
1. Evaluate 12 classification models (Tree-based, Linear, Distance, Neural Network).
2. Use **Optuna** to run 20 trials per model, optimizing for **F1-Score**.
3. Handle extreme class imbalance (65:1) using algorithmic weights.
4. Save models and predictions in an organized nested folder structure (`models/<name>/`, `outputs/<name>/`).
5. Track all metrics in a centralized log (`outputs/model_results_log.csv`).
6. Ensembling Phase: Pick top 5 models and find optimal soft voting weights via Optuna.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import optuna
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import f1_score, average_precision_score, accuracy_score, precision_score, recall_score

# Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

ModuleNotFoundError: No module named 'optuna'

In [ ]:
# Setup Directories
for dr in ['../models/from_notebook', '../outputs/from_notebook']:
    os.makedirs(dr, exist_ok=True)

# Load Data
try:
    df_train = pd.read_csv('../data/processed/train_engineered.csv')
    X = df_train.drop(columns=['is_fraud'])
    y = df_train['is_fraud']
    
    # Calculate Imbalance Ratio
    ratio = float(np.sum(y == 0)) / np.sum(y == 1)
    print(f"Loaded data: X {X.shape}, y {y.shape}")
    print(f"Class ratio (Negative/Positive): {ratio:.2f}")
except FileNotFoundError:
    print("Data not found. Ensure 02_Data_Preprocessing.ipynb has been run.")

In [ ]:
# Initialize Central Logger
log_path = '../outputs/from_notebook/model_results_log.csv'
if not os.path.exists(log_path):
    log_df = pd.DataFrame(columns=[
        'Model_Name', 'CV_Accuracy', 'CV_F1_Score', 'CV_PR_AUC', 
        'CV_Precision', 'CV_Recall', 'Best_Parameters', 
        'Model_File_Path', 'Output_Predictions_Path'
    ])
    log_df.to_csv(log_path, index=False)
else:
    log_df = pd.read_csv(log_path)

## Dynamic Optuna Pipeline

We define a general-purpose Optuna objective. The pipeline trains models via 5-Fold Stratified CV, logging the results of the best trial to our CSV tracker, and storing artifacts securely.

In [ ]:
def get_model_params(trial, model_name):
    """Returns hyperparameter search space and model class based on model_name."""
    if model_name == 'XGBoost':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'scale_pos_weight': ratio,
            'eval_metric': 'logloss',
            'random_state': 42
        }
        return XGBClassifier(**params)
        
    elif model_name == 'LightGBM':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'scale_pos_weight': ratio,
            'random_state': 42,
            'verbose': -1
        }
        return LGBMClassifier(**params)
        
    elif model_name == 'CatBoost':
        params = {
            'iterations': trial.suggest_int('iterations', 50, 300),
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'auto_class_weights': 'Balanced',
            'random_state': 42,
            'verbose': 0
        }
        return CatBoostClassifier(**params)
        
    elif model_name == 'RandomForest':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'class_weight': 'balanced',
            'random_state': 42
        }
        return RandomForestClassifier(**params)
        
    elif model_name == 'ExtraTrees':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'class_weight': 'balanced',
            'random_state': 42
        }
        return ExtraTreesClassifier(**params)
        
    elif model_name == 'GradientBoosting':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'random_state': 42
        }
        return GradientBoostingClassifier(**params)
        
    elif model_name == 'AdaBoost':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1.0, log=True),
            'random_state': 42
        }
        return AdaBoostClassifier(**params)
        
    elif model_name == 'LogisticRegression':
        params = {
            'C': trial.suggest_float('C', 1e-4, 10.0, log=True),
            'class_weight': 'balanced',
            'max_iter': 1000,
            'random_state': 42
        }
        return LogisticRegression(**params)
        
    elif model_name == 'SVC':
        params = {
            'C': trial.suggest_float('C', 1e-2, 10.0, log=True),
            'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf']),
            'class_weight': 'balanced',
            'probability': True,
            'random_state': 42
        }
        return SVC(**params)
        
    elif model_name == 'KNN':
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 3, 15),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance'])
        }
        return KNeighborsClassifier(**params)
        
    elif model_name == 'GaussianNB':
        params = {
            'var_smoothing': trial.suggest_float('var_smoothing', 1e-10, 1e-5, log=True)
        }
        return GaussianNB(**params)
        
    elif model_name == 'MLP':
        params = {
            'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(50,), (100,), (50, 25)]),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-1, log=True),
            'random_state': 42,
            'max_iter': 200
        }
        return MLPClassifier(**params)
    
    raise ValueError(f"Model {model_name} not supported.")

In [ ]:
def train_and_log_model(model_name, n_trials=20):
    print(f"\n{'='*40}\nOptimizing {model_name}\n{'='*40}")
    
    # Create Folders
    model_dir = f"../models/from_notebook/{model_name.lower()}"
    out_dir = f"../outputs/from_notebook/{model_name.lower()}"
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(out_dir, exist_ok=True)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    def objective(trial):
        model = get_model_params(trial, model_name)
        f1_scores = []
        for train_idx, val_idx in cv.split(X, y):
            X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            f1_scores.append(f1_score(y_val, preds))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)
    
    best_params = study.best_params
    print(f"Best CV F1-Score: {study.best_value:.4f}")
    
    # Train Best Model on full dataset to save
    print("Evaluating best parameters with Cross-Validation...")
    best_model = get_model_params(optuna.trial.FixedTrial(best_params), model_name)
    
    scoring = ['accuracy', 'f1', 'precision', 'recall']
    cv_res = cross_validate(best_model, X, y, cv=cv, scoring=scoring)
    
    # Ensure predict_proba exists before calling it
    if hasattr(best_model, 'predict_proba'):
        preds_proba = cross_val_predict(best_model, X, y, cv=cv, method='predict_proba')[:, 1]
    else:
        if hasattr(best_model, 'decision_function'):
            preds_proba = cross_val_predict(best_model, X, y, cv=cv, method='decision_function')
        else:
            preds_proba = cross_val_predict(best_model, X, y, cv=cv)
            
    pr_auc = average_precision_score(y, preds_proba)
    
    best_model.fit(X, y)
    
    model_path = f"{model_dir}/best_model.pkl"
    joblib.dump(best_model, model_path)
    
    preds_df = pd.DataFrame({'is_fraud': y, 'prediction_prob': preds_proba})
    preds_path = f"{out_dir}/predictions.csv"
    preds_df.to_csv(preds_path, index=False)
    
    metrics = {
        'Model_Name': model_name,
        'CV_Accuracy': np.mean(cv_res['test_accuracy']),
        'CV_F1_Score': np.mean(cv_res['test_f1']),
        'CV_PR_AUC': pr_auc,
        'CV_Precision': np.mean(cv_res['test_precision']),
        'CV_Recall': np.mean(cv_res['test_recall']),
        'Best_Parameters': str(best_params),
        'Model_File_Path': model_path,
        'Output_Predictions_Path': preds_path
    }
    
    log_df = pd.read_csv(log_path)
    log_df = pd.concat([log_df, pd.DataFrame([metrics])], ignore_index=True)
    log_df.to_csv(log_path, index=False)
    
    return best_model

## Train All Models

*Note: Running 20 trials for all 12 models can take significant time. Uncomment the loop to execute the exhaustive search.*

In [ ]:
models_to_test = [
    'LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 
    'CatBoost', 'ExtraTrees', 'GradientBoosting', 'AdaBoost',
    'GaussianNB', 'KNN', 'SVC', 'MLP'
]

# Uncomment to run the full pipeline
for m in models_to_test:
    train_and_log_model(m, n_trials=20)

## Optuna-Weighted Ensembling

Here we select the top 3 to 5 performing models from our log, load their Out-Of-Fold predictions, and use Optuna to find the optimal blending weights to maximize the ensemble's F1-Score.

In [ ]:
def build_ensemble():
    log_df = pd.read_csv(log_path)
    if len(log_df) == 0:
        print("No models logged yet.")
        return
        
    # Get top 5 models by F1, excluding previous ensembles
    top_models = log_df[log_df['Model_Name'] != 'Ensemble_SoftVoting'].sort_values(by='CV_F1_Score', ascending=False).head(5)
    print("Top 5 Models selected for Ensembling:")
    display(top_models[['Model_Name', 'CV_F1_Score']])
    
    oof_preds = []
    for path in top_models['Output_Predictions_Path']:
        df_p = pd.read_csv(path)
        oof_preds.append(df_p['prediction_prob'].values)
        
    oof_preds = np.array(oof_preds)
    true_labels = df_p['is_fraud'].values
    
    def ensemble_objective(trial):
        weights = [trial.suggest_float(f'w_{i}', 0, 1) for i in range(len(top_models))]
        weights = np.array(weights) / np.sum(weights)
        blended_probs = np.average(oof_preds, axis=0, weights=weights)
        blended_preds = (blended_probs >= 0.5).astype(int)
        return f1_score(true_labels, blended_preds)
        
    study = optuna.create_study(direction='maximize')
    study.optimize(ensemble_objective, n_trials=50, n_jobs=-1)
    
    best_weights = np.array([study.best_params[f'w_{i}'] for i in range(len(top_models))])
    best_weights = best_weights / np.sum(best_weights)
    print(f"\nBest Ensemble F1-Score: {study.best_value:.4f}")
    print(f"Optimal Weights: {best_weights}")
    
    ens_model_dir = "../models/from_notebook/ensemble"
    ens_out_dir = "../outputs/from_notebook/ensemble"
    os.makedirs(ens_model_dir, exist_ok=True)
    os.makedirs(ens_out_dir, exist_ok=True)
    
    blended_probs_final = np.average(oof_preds, axis=0, weights=best_weights)
    blended_preds_final = (blended_probs_final >= 0.5).astype(int)
    pr_auc = average_precision_score(true_labels, blended_probs_final)
    
    ens_preds_df = pd.DataFrame({'is_fraud': true_labels, 'prediction_prob': blended_probs_final})
    ens_preds_path = f"{ens_out_dir}/predictions.csv"
    ens_preds_df.to_csv(ens_preds_path, index=False)
    
    ens_meta = {'models': top_models['Model_File_Path'].tolist(), 'weights': best_weights.tolist()}
    joblib.dump(ens_meta, f"{ens_model_dir}/best_model.pkl")
    
    metrics = {
        'Model_Name': 'Ensemble_SoftVoting',
        'CV_Accuracy': accuracy_score(true_labels, blended_preds_final),
        'CV_F1_Score': f1_score(true_labels, blended_preds_final),
        'CV_PR_AUC': pr_auc,
        'CV_Precision': precision_score(true_labels, blended_preds_final),
        'CV_Recall': recall_score(true_labels, blended_preds_final),
        'Best_Parameters': str(ens_meta),
        'Model_File_Path': f"{ens_model_dir}/best_model.pkl",
        'Output_Predictions_Path': ens_preds_path
    }
    
    log_df = pd.read_csv(log_path)
    log_df = pd.concat([log_df, pd.DataFrame([metrics])], ignore_index=True)
    log_df.to_csv(log_path, index=False)
    print("Ensemble built and logged successfully.")

# Uncomment to build ensemble after training models
build_ensemble()